In [1]:
import pandas as pd

# 1. Load the raw file
df = pd.read_csv("../data/raw/crop_yield_raw.csv")
print("Raw:", df.shape)

# 2. Keep the columns we need and rename them
df = df[["REF_DATE", "GEO", "Type of crop", "UOM", "VALUE", "STATUS"]].copy()
df.columns = ["year", "province", "crop", "unit", "yield_kg_ha", "status"]

# 3. Keep one unit (the file has every yield in two units)
df = df[df["unit"] == "Kilograms per hectare"]

# 4. Keep three crops and simplify their names
crops = ["Barley", "Canola (rapeseed)", "Wheat, all"]
df = df[df["crop"].isin(crops)].copy()
df["crop"] = df["crop"].replace({"Canola (rapeseed)": "Canola", "Wheat, all": "Wheat"})

# 5. Keep provinces only (remove Canada and regional totals)
provinces = [
    "Newfoundland and Labrador", "Prince Edward Island", "Nova Scotia",
    "New Brunswick", "Quebec", "Ontario", "Manitoba", "Saskatchewan",
    "Alberta", "British Columbia",
]
df = df[df["province"].isin(provinces)].copy()

# 6b. Drop zero yields (a crop that was not really harvested)
print("Zero yields dropped:", (df["yield_kg_ha"] <= 0).sum())
df = df[df["yield_kg_ha"] > 0]

# 6c. Keep only crop-province series with at least 15 years of data,
# so averages and rankings are comparable
years_per_series = df.groupby(["crop", "province"])["year"].transform("count")
print("Rows dropped for sparse series:", (years_per_series < 15).sum())
df = df[years_per_series >= 15]

# 7. Final tidy
df["year"] = df["year"].astype(int)
df = df[["year", "province", "crop", "yield_kg_ha"]]
df = df.sort_values(["crop", "province", "year"]).reset_index(drop=True)

# 8. Checks
print("Clean:", df.shape)
print(df.groupby("crop")["year"].agg(["min", "max"]))
print(df.groupby("crop")["province"].nunique())
print("Duplicate rows:", df.duplicated(["year", "province", "crop"]).sum())

# 9. Save
df.to_csv("../data/clean/crop_yield_clean.csv", index=False)
df.head(10)

Raw: (5613, 16)
Zero yields dropped: 4
Rows dropped for sparse series: 10
Clean: (648, 4)
         min   max
crop              
Barley  2000  2026
Canola  2000  2026
Wheat   2000  2026
crop
Barley    9
Canola    6
Wheat     9
Name: province, dtype: int64
Duplicate rows: 0


,year,province,crop,yield_kg_ha
0,2000,Alberta,Barley,3000.0
1,2001,Alberta,Barley,2900.0
2,2002,Alberta,Barley,2300.0
3,2003,Alberta,Barley,3000.0
4,2004,Alberta,Barley,3500.0
5,2005,Alberta,Barley,3600.0
6,2006,Alberta,Barley,3200.0
7,2007,Alberta,Barley,3000.0
8,2008,Alberta,Barley,3600.0
9,2009,Alberta,Barley,3100.0
